# 02 — Feature Engineering (deterministic)

Derives new numeric features from the cleaned columns — the raw material every downstream model needs.
No ML yet: ratios, spreads, diversity/entropy, completeness, tiers, and interaction terms.
Reads `artifacts/localities_clean.parquet`, writes `artifacts/features_base.parquet`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ART = Path.cwd() / "artifacts"
df = pd.read_parquet(ART / "localities_clean.parquet")
print("Loaded:", df.shape)

NUM_COLS = [c for c in df.columns if c.startswith("num_")]
print("Amenity count columns:", NUM_COLS)

: 

In [2]:
# 1) Price-derived features
# buy/rent ratio ~ a rental-yield proxy (higher = pricier to buy vs rent)
df["buy_rent_ratio"] = df["res_avg_buy"] / df["res_avg_rent"]
# price spread = uncertainty / heterogeneity within the locality
df["price_spread"] = df["res_max_buy"] - df["res_min_buy"]
df["price_spread_pct"] = df["price_spread"] / df["res_avg_buy"]
# affluence tier from quantiles of avg buy price (labelled, NaN-safe)
df["affluence_tier"] = pd.qcut(df["res_avg_buy"], q=4,
                               labels=["Value", "Mid", "Premium", "Ultra-Premium"])
print(df[["res_avg_buy", "buy_rent_ratio", "price_spread", "affluence_tier"]].describe(include="all").to_string())

         res_avg_buy  buy_rent_ratio  price_spread affluence_tier
count     351.000000      351.000000    351.000000            351
unique           NaN             NaN           NaN              4
top              NaN             NaN           NaN          Value
freq             NaN             NaN           NaN             89
mean    13670.085470      417.642145   6262.108262            NaN
std      8326.828568      122.971128   3853.060394            NaN
min      3100.000000      193.333333   1600.000000            NaN
25%      7600.000000      326.166329   3500.000000            NaN
50%     11600.000000      396.491228   5300.000000            NaN
75%     16925.000000      488.508159   7250.000000            NaN
max     64700.000000     1297.619048  23400.000000            NaN


In [3]:
# 2) Amenity-derived features
# total amenities present (sum of de-duplicated counts)
df["total_amenities"] = df[NUM_COLS].sum(axis=1)

# amenity diversity = Shannon entropy across amenity types (0 = one type only, high = balanced mix)
def shannon_entropy(row):
    counts = np.array([row[c] for c in NUM_COLS], dtype=float)
    total = counts.sum()
    if total == 0:
        return 0.0
    p = counts[counts > 0] / total
    return float(-(p * np.log2(p)).sum())

df["amenity_diversity"] = df[NUM_COLS].apply(shannon_entropy, axis=1)

# infra completeness = share of the 7 amenity TYPES that are present (>=1)
df["infra_completeness"] = (df[NUM_COLS] > 0).sum(axis=1) / len(NUM_COLS)

# retail-vs-residential balance: shopping+commercial vs education+hospital (civic)
retail = df["num_shopping_centre"] + df["num_commercial_hub"]
civic = df["num_educational_institute"] + df["num_hospital"]
df["retail_civic_balance"] = (retail - civic) / (retail + civic).replace(0, np.nan)

print(df[["total_amenities", "amenity_diversity", "infra_completeness", "retail_civic_balance"]].describe().round(2).to_string())

       total_amenities  amenity_diversity  infra_completeness  retail_civic_balance
count           600.00             600.00              600.00                479.00
mean             11.79               1.27                0.46                 -0.36
std               7.66               0.80                0.28                  0.59
min               0.00               0.00                0.00                 -1.00
25%               8.00               0.54                0.29                 -1.00
50%              11.00               1.31                0.43                 -0.43
75%              16.00               1.93                0.71                  0.00
max              38.00               2.73                1.00                  1.00


In [4]:
# 3) Interaction features (give downstream models non-linear combinations up front)
# Use median-imputed, scaled copies ONLY for interactions so NaNs don't propagate everywhere.
for base in ["res_avg_buy", "total_amenities", "infra_completeness", "num_transportation_hub"]:
    z = (df[base] - df[base].mean()) / df[base].std(ddof=0)
    df[base + "_z"] = z

df["affluence_x_amenities"] = df["res_avg_buy_z"] * df["total_amenities_z"]
df["transport_x_infra"] = df["num_transportation_hub_z"] * df["infra_completeness_z"]
print("Interaction features added:",
      [c for c in df.columns if c.endswith("_z") or "_x_" in c])

Interaction features added: ['res_avg_buy_z', 'total_amenities_z', 'infra_completeness_z', 'num_transportation_hub_z', 'affluence_x_amenities', 'transport_x_infra']


In [5]:
# 4) Quality / sanity checks before saving
n = len(df)
new_feats = ["buy_rent_ratio", "price_spread", "price_spread_pct", "affluence_tier",
             "total_amenities", "amenity_diversity", "infra_completeness",
             "retail_civic_balance", "affluence_x_amenities", "transport_x_infra"]
report = pd.DataFrame({
    "non_null": [df[c].notna().sum() for c in new_feats],
    "pct_present": [round(df[c].notna().mean() * 100, 1) for c in new_feats],
}, index=new_feats)
print("New feature coverage:")
print(report.to_string())

# correlation of the numeric new features with affluence (where present)
corr = df[["res_avg_buy", "total_amenities", "amenity_diversity",
           "infra_completeness", "num_transportation_hub"]].corr()["res_avg_buy"].round(2)
print("\nCorrelation with res_avg_buy:")
print(corr.to_string())

New feature coverage:
                       non_null  pct_present
buy_rent_ratio              351         58.5
price_spread                351         58.5
price_spread_pct            351         58.5
affluence_tier              351         58.5
total_amenities             600        100.0
amenity_diversity           600        100.0
infra_completeness          600        100.0
retail_civic_balance        479         79.8
affluence_x_amenities       351         58.5
transport_x_infra           600        100.0

Correlation with res_avg_buy:
res_avg_buy               1.00
total_amenities           0.08
amenity_diversity         0.07
infra_completeness        0.06
num_transportation_hub    0.20


In [6]:
# 5) Persist
out = ART / "features_base.parquet"
df.to_parquet(out, index=False)
print("Saved", df.shape[0], "rows x", df.shape[1], "cols ->", out.relative_to(Path.cwd()))

Saved 600 rows x 45 cols -> artifacts\features_base.parquet


## Output
`artifacts/features_base.parquet` — adds price ratios/spread/tier, amenity diversity (entropy),
infra completeness, retail-vs-civic balance, and interaction terms.

**Next:** `03_text_mining.ipynb` — NER, keyword/sector tagging, sentence embeddings, and topic modelling
on the five free-text columns (Path A: sentence-transformers + zero-shot + BERTopic).